In [2]:
from datetime import datetime, timedelta

import numpy as np
from scipy.signal import get_window
from stonesoup.models.transition.linear import (
    CombinedLinearGaussianTransitionModel,
    ConstantVelocity,
)
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState

from nereus.detector import CFARDetector, PassiveSonarDetector, PeakDetector
from nereus.models.environment import FlatBathymetry, Linear
from nereus.models.propagation import CylindricalAcousticPropagationModel
from nereus.platform import TowedArrayPlatform
from nereus.signal.ambient import ColouredNoise
from nereus.signal.anthropogenic import BroadbandShipSignal
from nereus.sigproc import (
    DelayAndSumBeamformer,
    MinimumVarianceDistortionlessResponseBeamformer,
    SteeringCalculator,
)
from nereus.simulator import BroadbandPassiveSonarArraySimulator

seed = 2000
np.random.seed(seed)

# SIMULATION PARAMETERS

In [3]:
# ============================================================================
# SIMULATION PARAMETERS
# ============================================================================

SIM_LENGTH = 900 # seconds
SIM_RATE = 5.0 # seconds

Xlim = [-15_000, 15_000]
Ylim = [-15_000, 15_000]

NUM_TARGETS = 1

SIM_PARAMS = {
    "start_time":datetime.now().replace(hour=0, minute=0, second=0, microsecond=0),
    "time_interval":timedelta(seconds=SIM_RATE),
    "num_steps":int(SIM_LENGTH / SIM_RATE),
}

total_duration_s = SIM_PARAMS["num_steps"] * SIM_PARAMS["time_interval"].total_seconds()
print(f"Total simulation duration: {total_duration_s} seconds")
print(f"Number of timesteps: {SIM_PARAMS['num_steps']}, "
      f"Timestep interval: {SIM_PARAMS['time_interval'].total_seconds()} seconds")

SHIP_PARAMS = {
    "start_vector": np.array([-2000.0, 5.0, 2000.0, 0.0, -5.0, 0.0]),
    "position_mapping": [0, 2, 4],
    "velocity_mapping": [1, 3, 5],
    "transition_model": CombinedLinearGaussianTransitionModel(
        [ConstantVelocity(0), ConstantVelocity(0), ConstantVelocity(0)]
    ),
}

ARRAY_PARAMS = {
    "num_sensors": 50,
    "tow_cable_length": 100.0,
    "sensor_spacing": 0.5,
    "array_depth": -50.0,
}

TARGETS = []

for _ in range(NUM_TARGETS):
    TARGET_PARAMS = {
        "start_vector": np.array([0.0, 0.0, 0.0, 8.0, -5.0, 0.0]),
        "position_mapping": [0, 2, 4],
        "velocity_mapping": [1, 3, 5],
        "transition_model": CombinedLinearGaussianTransitionModel(
            [ConstantVelocity(0), ConstantVelocity(0), ConstantVelocity(0)]
        ),
        "amplitudes_upa": 10 ** (np.random.uniform(87, 102, 4) / 20),
        "frequencies_hz": np.random.uniform(25.0, 200.0, 4),
        "phases_rad": np.random.uniform(0, 2 * np.pi, 4),
        "tonal_bandwidth_hz": np.random.uniform(0.5, 2.0),
        "noise_amplitude_upa":10 ** (np.random.uniform(65, 85) / 20),
        "noise_spectral_exponent":-1.0,
    }
    TARGETS.append(TARGET_PARAMS)

SIGNAL_PARAMS = {
    "duration_s": total_duration_s,
    "sampling_rate_hz": 500.0,
    "frame_len": 500,
    "hop_factor": 2,
    "fade_in_ms": 1000.0,
}

AMBIENT_NOISE_PARAMS = {
    "amplitude_upa": 10 ** (np.random.uniform(45, 55) / 20),
    "spectral_exponent": -1,
}

PROP_PARAMS = {
    "ssp": Linear(surface_speed=1500.0, gradient=0.2),
    "attenuation_factor": 0.5, 
    "bathymetry": FlatBathymetry(depth=-150.0),
    "step_m": 20.0,
    "azimuth_search_width": 2.0,
    "azimuth_resolution": 0.5,
    "elevation_range": (-25.0, 25.0),
    "elevation_resolution": 1.0,
}

BF_PARAMS = {
    "beamformer_type": "MVDR",
    "shading": None,    # shading is only for DAS, does nothing with MVDR
    "domain": "broadband_power",  # "time", "frequency", or "broadband_power" for DAS
    "steering_azimuths_rad": np.linspace(-np.pi, np.pi, 181), 
    "fmin": 100.0,
    "fmax": 125.0,
}

# Detection parameters
DET_PARAMS = {
    "cfar_detector": {
        "num_guard_cells": 2,
        "num_training_cells": 8,
        "threshold_factor": 1.95,
    },
    "peak_detector": {
        "distance": 3,
    }
}

Total simulation duration: 900.0 seconds
Number of timesteps: 180, Timestep interval: 5.0 seconds


# SETUP

In [4]:
# ============================================================================
# SETUP
# ============================================================================

# Create platform
initial_state = GroundTruthState(
    SHIP_PARAMS["start_vector"], timestamp=SIM_PARAMS["start_time"]
)

platform = TowedArrayPlatform(
    states=initial_state,
    position_mapping=[0, 2, 4],
    velocity_mapping=[1, 3, 5],
    transition_models=[SHIP_PARAMS["transition_model"]],
    transition_times=[timedelta(seconds=SIM_LENGTH)],
    num_sensors=ARRAY_PARAMS["num_sensors"],
    cable_length_m=ARRAY_PARAMS["tow_cable_length"],
    sensor_spacing_m=ARRAY_PARAMS["sensor_spacing"],
    array_depth_m=ARRAY_PARAMS["array_depth"],
)

for i in range(1, SIM_PARAMS["num_steps"]):
    new_time = SIM_PARAMS["start_time"] + i * SIM_PARAMS["time_interval"]
    platform.move(new_time)

# Create target trajectories
target_ground_truths = []
relative_bearing_ground_truths = []

for TARGET_PARAMS in TARGETS:

    # Create initial state
    target_states = [
        GroundTruthState(
            TARGET_PARAMS["start_vector"],
            timestamp=SIM_PARAMS["start_time"],
            metadata={
                "amplitudes_upa": TARGET_PARAMS["amplitudes_upa"],
                "frequencies_hz": TARGET_PARAMS["frequencies_hz"],
                "phases_rad": TARGET_PARAMS["phases_rad"],
                "position_mapping": TARGET_PARAMS["position_mapping"],
                "velocity_mapping": TARGET_PARAMS["velocity_mapping"],
                "tonal_bandwidth_hz": TARGET_PARAMS["tonal_bandwidth_hz"],
                "noise_amplitude_upa": TARGET_PARAMS["noise_amplitude_upa"],
                "noise_spectral_exponent": TARGET_PARAMS["noise_spectral_exponent"],
            },
        )
    ]

    # Propagate trajectory with multi-transition models
    current_maneuver_idx = 0
    for i in range(1, SIM_PARAMS["num_steps"]):

        transition_model = TARGET_PARAMS["transition_model"]
        new_time = SIM_PARAMS["start_time"] + i * SIM_PARAMS["time_interval"]
        time_interval = new_time - target_states[-1].timestamp
        new_state_vector = transition_model.function(
            target_states[-1], noise=False, time_interval=time_interval
        )
        new_state = GroundTruthState(
            new_state_vector,
            timestamp=new_time,
            metadata=target_states[-1].metadata,
        )
        target_states.append(new_state)

    target_ground_truth = GroundTruthPath(target_states)
    target_ground_truths.append(target_ground_truth)

    # Compute ground truth bearings for plotting
    gt_relative_bearings = []
    for target_state in target_ground_truth.states:
        platform_state = platform.get_platform_state_at(target_state.timestamp)
        ref_sensor_position = np.mean(platform_state.array.state_vector, axis=1)
        target_pos = np.array([target_state.state_vector[0], target_state.state_vector[2]])
        relative_pos = target_pos - ref_sensor_position[:2]
        bearing_rad = np.arctan2(relative_pos[1], relative_pos[0])
        gt_relative_bearings.append(bearing_rad)

    gt_relative_bearings = np.array(gt_relative_bearings)

    # Create GroundTruthPath for bearings
    relative_bearing_truth_states = []
    for i, bearing in enumerate(gt_relative_bearings):
        timestamp = SIM_PARAMS["start_time"] + i * SIM_PARAMS["time_interval"]
        bearing_state = GroundTruthState(
            state_vector=np.array([bearing]),
            timestamp=timestamp
        )
        relative_bearing_truth_states.append(bearing_state)

    relative_bearing_ground_truth = GroundTruthPath(relative_bearing_truth_states)
    relative_bearing_ground_truths.append(relative_bearing_ground_truth)

# Create signal and propagation models

prop_model = CylindricalAcousticPropagationModel(
    ssp=PROP_PARAMS["ssp"],
    attenuation_factor=PROP_PARAMS["attenuation_factor"],
)

# Create ambient noise model
ambient_noise_model = ColouredNoise(
    amplitude_upa=AMBIENT_NOISE_PARAMS["amplitude_upa"],
    spectral_exponent=AMBIENT_NOISE_PARAMS["spectral_exponent"],
    duration_s=SIM_PARAMS["time_interval"].total_seconds(),
    sampling_rate_hz=SIGNAL_PARAMS["sampling_rate_hz"],
)

# Create beamformer and steering calculator
shading = None
if BF_PARAMS["shading"] is not None:
    shading = get_window(BF_PARAMS["shading"], platform.num_sensors)


if BF_PARAMS["beamformer_type"] == "DAS":
    if BF_PARAMS["domain"] == "broadband_power":
        beamformer = DelayAndSumBeamformer(
            domain=BF_PARAMS["domain"],
            sampling_rate_hz=SIGNAL_PARAMS["sampling_rate_hz"],
            fmin=BF_PARAMS["fmin"],
            fmax=BF_PARAMS["fmax"],
        )
    else:
        beamformer = DelayAndSumBeamformer(
            sampling_rate_hz=SIGNAL_PARAMS["sampling_rate_hz"],
            shading=shading,
            domain=BF_PARAMS["domain"]
        )
elif BF_PARAMS["beamformer_type"] == "MVDR":

    beamformer = MinimumVarianceDistortionlessResponseBeamformer(
        sampling_rate_hz=SIGNAL_PARAMS["sampling_rate_hz"],
        fmin=BF_PARAMS["fmin"],
        fmax=BF_PARAMS["fmax"],
    )
else:
    raise ValueError(f"Unknown beamformer type: {BF_PARAMS['beamformer_type']}")

steering_calculator = SteeringCalculator(
    ssp=PROP_PARAMS["ssp"],
    steering_azimuths_rad=BF_PARAMS["steering_azimuths_rad"],
)

# Create signal models for each target
signal_models = []
for TARGET_PARAMS in TARGETS:
    signal_model = BroadbandShipSignal(
        duration_s=SIGNAL_PARAMS["duration_s"],
        sampling_rate_hz=SIGNAL_PARAMS["sampling_rate_hz"],
        frame_len=SIGNAL_PARAMS["frame_len"],
        hop_factor=SIGNAL_PARAMS["hop_factor"],
        tonal_bandwidth_hz=TARGET_PARAMS["tonal_bandwidth_hz"],
        noise_amplitude_upa=TARGET_PARAMS["noise_amplitude_upa"],
        noise_spectral_exponent=TARGET_PARAMS["noise_spectral_exponent"],
        noise_freq_range_hz=(0.0, SIGNAL_PARAMS["sampling_rate_hz"]/2),
        tonal_noise_is_constant=True,
        noise_is_constant=True,
    )
    signal_models.append(signal_model)

# Use the first signal model for simplicity (have same time/freq params but different
# source signal)
signal_model = signal_models[0]

# Create simulator with beamforming
simulator = BroadbandPassiveSonarArraySimulator(
    platform=platform,
    propagation_model=prop_model,
    signal_model=signal_model,
    noise_model=ambient_noise_model,
    beamformer=beamformer,
    steering_calculator=steering_calculator,
    ground_truth_paths=target_ground_truths,
    fade_in_ms=SIGNAL_PARAMS["fade_in_ms"],
)

# RUN SIMULATION WITH DETECTION

In [5]:
# ============================================================================
# RUN SIMULATION WITH DETECTION
# ============================================================================

data_generator = simulator.sensor_data_gen()

# Create detection chain
cfar_detector = CFARDetector(
    num_guard_cells=DET_PARAMS["cfar_detector"]["num_guard_cells"],
    num_training_cells=DET_PARAMS["cfar_detector"]["num_training_cells"],
    threshold_factor=DET_PARAMS["cfar_detector"]["threshold_factor"],
)


detection_chain = [cfar_detector]
if DET_PARAMS["peak_detector"]["distance"] > 0:
    peak_detector = PeakDetector(distance=DET_PARAMS["peak_detector"]["distance"])
    detection_chain.append(peak_detector)

detector = PassiveSonarDetector(
    detection_chain=detection_chain,
    sensor_data_gen=data_generator,
    steering_azimuths_rad=BF_PARAMS["steering_azimuths_rad"]
)

# Run detection
print("Running detection chain...")
all_detections = list(detector.detections_gen(progress_bar=True))
snr_map = detector.snr_history

Running detection chain...


Generating Detections: 180it [00:18,  9.75it/s]


# VISUALISATION

In [6]:
import plotly.graph_objects as go

timesteps = [
    SIM_PARAMS["start_time"] + i * SIM_PARAMS["time_interval"]
    for i in range(SIM_PARAMS["num_steps"])
]

fig = go.Figure()

plat_x = []
plat_y = []

print(len(timesteps))

for timestamp in timesteps:
    platform_state = platform.get_platform_state_at(timestamp)
    plat_x.append(platform_state.host.state.state_vector[0] / 1000) # convert to km
    plat_y.append(platform_state.host.state.state_vector[2] / 1000)

tgt_x = []
tgt_y = []

for target_ground_truth in target_ground_truths:
    tgt_x = [state.state_vector[0] / 1000 for state in target_ground_truth]
    tgt_y = [state.state_vector[2] / 1000 for state in target_ground_truth]

# --- 2. Plot the Lines ---
fig.add_trace(go.Scatter(
    x=plat_x, y=plat_y,
    mode='lines', line=dict(color='black', width=3),
    name='Platform'
))

fig.add_trace(go.Scatter(
    x=tgt_x, y=tgt_y,
    mode='lines', line=dict(color="red", width=3, dash="5px,2px"),
    name='Target'
))

# --- 3. Arrow Function ---
def add_arrowhead_at_index(fig, x_arr, y_arr, i, color, size=0.1):
    """Add a triangle (arrowhead) at index 'i' pointing in the direction of travel.

    Parameters
    ----------
    fig: The Plotly figure to add the shape to.
    x_arr: Array of x coordinates.
    y_arr: Array of y coordinates.
    i: Index at which to place the arrowhead (must be >= 1).
    color: The fill color.
    size: The length of the arrowhead in data units (e.g., kilometers).

    """
    # Boundary check
    if i < 1 or i >= len(x_arr):
        return

    # 1. Calculate Direction Vector (u)
    dx = x_arr[i] - x_arr[i-1]
    dy = y_arr[i] - y_arr[i-1]
    magnitude = np.sqrt(dx**2 + dy**2)

    if magnitude > 0:
        # Normalized direction vector
        ux = dx / magnitude
        uy = dy / magnitude

        # Perpendicular vector (v) for the width
        vx, vy = -uy, ux

        # 2. Define Arrow Geometry
        # Tip Position (with offset applied)
        tip_x = x_arr[i]
        tip_y = y_arr[i]

        # Base Center Position (behind the tip)
        base_center_x = tip_x - (ux * size)
        base_center_y = tip_y - (uy * size)

        # Width of the arrow head (e.g., 70% of the length)
        half_width = size * 0.35

        # Calculate Left and Right corners of the base
        left_x = base_center_x + (vx * half_width)
        left_y = base_center_y + (vy * half_width)

        right_x = base_center_x - (vx * half_width)
        right_y = base_center_y - (vy * half_width)

        # 3. Construct SVG Path
        # Move to Tip -> Line to Left -> Line to Right -> Close (Z)
        path = (f"M {tip_x},{tip_y} "
                f"L {left_x},{left_y} "
                f"L {right_x},{right_y} "
                "Z")

        # 4. Add Shape
        fig.add_shape(
            type="path",
            path=path,
            fillcolor=color,
            line=dict(
                color=color,  # Outline Color
                width=1       # Outline Thickness
            ),
            xref="x", yref="y"  # Use data coordinates
        )

# --- 4. Loop to Add Arrows ---

# Add GREEN arrows every 30th step
# Start at 10 to ensure we have previous history
for i in range(10, len(plat_x), 30):
    add_arrowhead_at_index(fig, plat_x, plat_y, i, "black", size=0.3)

# Add RED arrows every 30th step
for i in range(10, len(tgt_x), 30):
    add_arrowhead_at_index(fig, tgt_x, tgt_y, i, "red", size=0.3)

# --- 5. Layout Update ---
fig.update_layout(
    width=600, height=600,
    font=dict(family="Times New Roman", size=16, color="black"),
    showlegend=True,
    legend=dict(x=0.5, y=-0.25, xanchor="center", orientation="h", yanchor="bottom"),
    plot_bgcolor="white",
    yaxis=dict(
        scaleanchor="x",
        scaleratio=1
    )
)

# Calculate ranges with padding
if len(plat_x) > 0 and len(tgt_x) > 0:
    all_x = plat_x + tgt_x
    all_y = plat_y + tgt_y
    xrange = [min(all_x) - 0.5, max(all_x) + 0.5]
    yrange = [min(all_y) - 0.5, max(all_y) + 0.5]

    fig.update_xaxes(
        range=xrange, showgrid=True, gridcolor="rgba(200, 200, 200, 0.5)",
        linecolor="black", title="X Position (km)", zeroline=True,
        zerolinecolor="rgba(200, 200, 200, 0.5)", zerolinewidth=0.5
    )

    fig.update_yaxes(
        range=yrange, showgrid=True, gridcolor="rgba(200, 200, 200, 0.5)",
        linecolor="black", title="Y Position (km)", zeroline=True,
        zerolinecolor="rgba(200, 200, 200, 0.5)", zerolinewidth=0.5
    )

fig.show()

fig.write_image("figs/st_world_picture.pdf", scale=1, width=600, height=600)

180


# TARGET TRACKING

In [7]:
# ============================================================================
# TARGET TRACKING
# ============================================================================

from stonesoup.dataassociator.probability import PDA
from stonesoup.functions import (
    gm_reduce_single,  # For merging states to get posterior estimate
)
from stonesoup.hypothesiser.probability import PDAHypothesiser
from stonesoup.models.measurement.linear import LinearGaussian
from stonesoup.models.transition.linear import ConstantVelocity
from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.types.array import StateVectors
from stonesoup.types.state import GaussianState
from stonesoup.types.track import Track
from stonesoup.types.update import GaussianStateUpdate  # To store posterior estimate
from stonesoup.updater.kalman import ExtendedKalmanUpdater

# 1. Transition Model and Predictor
transition_model = ConstantVelocity(0.000001)
predictor = KalmanPredictor(transition_model)

# 2. Measurement Model and Updater
measurement_model = LinearGaussian(
    ndim_state=2,
    mapping=[0],
    noise_covar=np.array([[np.deg2rad(1)**2]])
)
updater = ExtendedKalmanUpdater(measurement_model=measurement_model)

# 3. Hypothesiser and Data Associator
hypothesiser = PDAHypothesiser(
    predictor=predictor,
    updater=updater,
    clutter_spatial_density=5/np.pi,
    prob_detect=0.95
)
data_associator = PDA(hypothesiser=hypothesiser)

# 5. Manually Create Initial Track
# Get the initial ground truth bearing and add some noise
initial_bearing = relative_bearing_ground_truth[0].state_vector[0] + \
    np.random.normal(0, np.deg2rad(2))
initial_state_vector = np.array([initial_bearing, 0]) # Assume initial bearing rate is 0

# Define initial state with uncertainty
prior_state = GaussianState(
    initial_state_vector,
    np.diag([np.deg2rad(5)**2, np.deg2rad(0.5)**2]),
    timestamp=SIM_PARAMS["start_time"]
)

# Create a track object
track = Track([prior_state])

# 7. Run Tracker
for timestamp, detections in all_detections:
    hypotheses = data_associator.associate(
        {track},
        detections,
        timestamp
    )

    hypotheses = hypotheses[track]

    posterior_states = []
    posterior_state_weights = []
    for hypothesis in hypotheses:
        if not hypothesis:
            posterior_states.append(hypothesis.prediction)
        else:
            posterior_state = updater.update(hypothesis)
            posterior_states.append(posterior_state)
        posterior_state_weights.append(hypothesis.probability)

    means = StateVectors([state.state_vector for state in posterior_states])
    covars = np.stack([state.covar for state in posterior_states], axis=2)
    weights = np.asarray(posterior_state_weights)

    # Reduce mixture of states to one posterior Gaussian estimate
    post_mean, post_covar = gm_reduce_single(means, covars, weights)

    # Add a Gaussian state approximation to the track
    track.append(
        GaussianStateUpdate(
            post_mean, post_covar,
            hypotheses,
            hypotheses[0].measurement.timestamp
        )
    )

In [8]:
from plotly.subplots import make_subplots

det_x = []
det_y = []

for _, detections in all_detections:
    for det in detections:
        det_x.append(np.rad2deg(det.state_vector[0]))
        det_y.append(det.timestamp)

track_x = [np.rad2deg(state.state_vector[0]) for state in track]
track_y = [state.timestamp for state in track]

gt_x = [np.rad2deg(state.state_vector[0]) for state in relative_bearing_ground_truth]
gt_y = [state.timestamp for state in relative_bearing_ground_truth]

fig = make_subplots(
    rows=1, cols=3, shared_xaxes=True, shared_yaxes=True, horizontal_spacing=0.06,
    subplot_titles=["(a)", "(b)", "(c)"]
)

fig.add_trace(
    go.Heatmap(
        z=snr_map,
        y=timesteps,
        x=np.rad2deg(BF_PARAMS["steering_azimuths_rad"]),
        colorscale="Viridis",
        colorbar=dict(
            title=dict(text="SNR (dB)", side="right", font=dict(size=16)),
            thickness=24,
            len=1.0,
            tickfont=dict(size=14),
        ),
    ),
    row=1, col=1
)

fig.add_trace(
    go.Heatmap(
        z=snr_map,
        y=timesteps,
        x=np.rad2deg(BF_PARAMS["steering_azimuths_rad"]),
        colorscale="Viridis",
        showscale=False
    ),
    row=1, col=2
)

fig.add_trace(
    go.Scatter(
        x=det_x,
        y=det_y,
        mode="markers",
        name="Detection",
        marker=dict(
            size=6,
            line=dict(width=1),
            color="white",
            opacity=1.0,
        ),
        hovertemplate="Bearing: %{x:.1f}°<br>Time: %{y|%H:%M:%S}<extra></extra>",
    ),
    row=1, col=2
)

fig.add_trace(
    go.Scatter(
        x=det_x,
        y=det_y,
        mode="markers",
        name="Detection",
        showlegend=False,
        marker=dict(
            size=6,
            line=dict(width=1),
            color="white",
            opacity=0.8,
        ),
        hovertemplate="Bearing: %{x:.1f}°<br>Time: %{y|%H:%M:%S}<extra></extra>",
    ),
    row=1, col=3
)

fig.add_trace(
    go.Scatter(
        x=track_x,
        y=track_y,
        mode="lines",
        name="Track",
        line=dict(color="blue", width=4),
        hovertemplate="Bearing: %{x:.1f}°<br>Time: %{y|%H:%M:%S}<extra></extra>",
    ),
    row=1, col=3
)

fig.add_trace(
    go.Scatter(
        x=gt_x,
        y=gt_y,
        mode="lines",
        name="Ground Truth",
        line=dict(color="red", width=3, dash="dash"),
        hovertemplate="Bearing: %{x:.1f}°<br>Time: %{y|%H:%M:%S}<extra></extra>",
    ),
    row=1, col=3
)

fig.update_xaxes(
    range=[-180, 180],
    tickmode="linear",
    tick0=-180,
    dtick=60,
    tickangle=-45,
    tickfont=dict(size=14),
    showgrid=True,
    gridcolor="rgba(200, 200, 200, 0.5)",
    title="Bearing (°)",
    ticks="outside",
    tickcolor="rgba(160, 160, 160, 1.0)"
)
fig.update_xaxes(title="", col=1)
fig.update_xaxes(
    title="",
    col=3,
    showline=True,
    linewidth=1,
    linecolor="rgba(160, 160, 160, 1.0)"
)

fig.update_yaxes(
    range=[timesteps[-1], timesteps[0]],
    showgrid=True,
    gridcolor="rgba(200, 200, 200, 0.5)",
    tickformat="%H:%M",
    tickfont=dict(size=14),
    autorange=False,
    title="Time (HH:MM)",
    tickcolor="rgba(160, 160, 160, 1.0)"
)
fig.update_yaxes(ticks="outside", col=1)
fig.update_yaxes(showticklabels=False, title="", col=2)
fig.update_yaxes(
    showticklabels=False,
    title="",
    col=3,
    showline=True,
    linewidth=1,
    linecolor="rgba(160, 160, 160, 1.0)"
)

fig.update_layout(
    width=1200,
    height=600,
    margin=dict(b=100),
    font=dict(family="Times New Roman", size=16, color="black"),
    showlegend=True,
    legend=dict(
        x=0.5,
        y=-0.3,
        xanchor="center",
        yanchor="bottom",
        bgcolor="rgba(255,255,255,0.0)",
        borderwidth=0,
        orientation="h"
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
)

fig.show()

fig.write_image("figs/st_bf_tracker.pdf", scale=1, width=1200, height=600)

# STONE SOUP DETECTIONS

In [9]:
# ============================================================================
# STONE SOUP DECTIONS
# ============================================================================
from scipy.stats import uniform
from stonesoup.types.detection import Detection

# Measurement model for detection generation (1D state -> 1D measurement)
deg_std = 0.5
detection_measurement_model = LinearGaussian(
    ndim_state=1,
    mapping=[0],
    noise_covar=np.array([[np.deg2rad(deg_std)**2]])
)

# Measurement model for tracking (2D state -> 1D measurement)
ss_measurement_model = LinearGaussian(
    ndim_state=2,
    mapping=[0],
    noise_covar=np.array([[np.deg2rad(deg_std)**2]])
)

# Detection parameters
FOV_RAD = np.deg2rad(360)
expected_false_alarms_per_scan = 1
clutter_spatial_density = expected_false_alarms_per_scan / FOV_RAD
print(f"Using clutter spatial density: {clutter_spatial_density:.4f} per radian")
print(f"Using clutter spatial density: {expected_false_alarms_per_scan/360:.4f} per degree")
prob_detection = 0.95

# Generate detections from ground truth
stone_soup_detections = []

for i, timestamp in enumerate(timesteps):
    detections_at_time = []

    # Generate true detections from each target
    for gt_bearing_path in relative_bearing_ground_truths:
        if np.random.rand() < prob_detection:
            # Use measurement model to generate noisy measurement
            measurement = detection_measurement_model.function(
                gt_bearing_path[i], noise=True
            )

            detection = Detection(
                state_vector=measurement,
                timestamp=timestamp,
                measurement_model=ss_measurement_model
            )
            detections_at_time.append(detection)
            detection_mirrored = Detection(
                state_vector=-measurement,
                timestamp=timestamp,
                measurement_model=ss_measurement_model
            )
            detections_at_time.append(detection_mirrored)

    # Add clutter/false alarms
    num_clutter = np.random.poisson(clutter_spatial_density)
    for _ in range(num_clutter):
        clutter_bearing = uniform.rvs(loc=-np.pi, scale=2*np.pi)
        clutter_detection = Detection(
            state_vector=np.array([[clutter_bearing]]),
            timestamp=timestamp,
            measurement_model=ss_measurement_model
        )
        detections_at_time.append(clutter_detection)

    stone_soup_detections.append((timestamp, detections_at_time))

# Convert to format compatible with plotter
ss_detections_for_plotter = [detection_set for _, detection_set in stone_soup_detections]

Using clutter spatial density: 0.1592 per radian
Using clutter spatial density: 0.0028 per degree


In [10]:
fig = make_subplots(
    rows=1, cols=2, shared_xaxes=True, shared_yaxes=True, horizontal_spacing=0.06,
    subplot_titles=["(a)", "(b)"]
)

fig.add_trace(
    go.Scatter(
        x=det_x,
        y=det_y,
        mode="markers",
        name="Detection",
        showlegend=False,
        marker=dict(
            size=6,
            line=dict(width=0.5),
            color="white",
            opacity=0.8,
        ),
        hovertemplate="Bearing: %{x:.1f}°<br>Time: %{y|%H:%M:%S}<extra></extra>",
    ),
    row=1, col=1
)

ss_det_x = []
ss_det_y = []

for t, detection_set in stone_soup_detections:
    for detection in detection_set:
        ss_det_x.append(np.rad2deg(detection.state_vector[0]))
        ss_det_y.append(t)

fig.add_trace(
    go.Scatter(
        x=ss_det_x,
        y=ss_det_y,
        mode="markers",
        name="Detection",
        showlegend=True,
        marker=dict(
            size=6,
            line=dict(width=0.5),
            color="white",
            opacity=0.8,
        ),
        hovertemplate="Bearing: %{x:.1f}°<br>Time: %{y|%H:%M:%S}<extra></extra>",
    ),
    row=1, col=2
)

fig.update_xaxes(
    range=[-180, 180],
    tickmode="linear",
    tick0=-180,
    dtick=60,
    tickangle=-45,
    tickfont=dict(size=14),
    ticks="outside",
    tickcolor="rgba(160, 160, 160, 1.0)",
    showgrid=True,
    gridcolor="rgba(200, 200, 200, 0.5)",
    showline=True,
    linecolor="rgba(160, 160, 160, 1.0)",
)
fig.add_annotation(
    text="Bearing (°)",
    xref="paper", yref="paper",
    x=0.5,
    y=-0.18,
    showarrow=False,
    font=dict(family="Times New Roman", size=18, color="black")
)

fig.update_yaxes(
    range=[timesteps[-1], timesteps[0]],
    tickformat="%H:%M",
    tickfont=dict(size=16),
    ticks="outside",
    tickcolor="rgba(160, 160, 160, 1.0)",
    showgrid=True,
    gridcolor="rgba(200, 200, 200, 0.5)",
    showline=True,
    linewidth=1,
    linecolor="rgba(160, 160, 160, 1.0)",
    autorange=False,
    title="Time (HH:MM)"
)
fig.update_yaxes(title="", ticks="", showticklabels=False, col=2)

fig.update_layout(
    width=600,
    height=600,
    margin=dict(b=100),
    font=dict(family="Times New Roman", size=16, color="black"),
    showlegend=False,
    legend=dict(
        x=1.0,
        y=1.02,
        xanchor="right",
        yanchor="bottom",
        bgcolor="rgba(255,255,255,0.0)",
        borderwidth=0,
        orientation="h"
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
)

fig.show()

fig.write_image("figs/st_plugin_vs_ss.pdf", scale=1, width=600, height=600)

In [11]:
# ============================================================================
# PRINT SUMMARY
# ============================================================================
tab = " " * 4

print()
print("=" * 80)
print("Random Scenario Summary:")
print("=" * 80)
print(f"Seed: {seed}")
print(f"Number of Targets: {NUM_TARGETS}")
print(f"Simulation Duration: {total_duration_s} s "
      f"across {SIM_PARAMS['num_steps']} timesteps")
print(f"Timestep Interval: {SIM_PARAMS['time_interval'].total_seconds()} s")
print(f"Array: {ARRAY_PARAMS['num_sensors']} sensors, "
      f"{ARRAY_PARAMS['sensor_spacing']} m spacing")
print(f"Signal Sampling Rate: {SIGNAL_PARAMS['sampling_rate_hz']} Hz")
print(f"Ambient Noise Level: {20 * np.log10(AMBIENT_NOISE_PARAMS['amplitude_upa']):.1f}"
      f" dB re 1 µPa")
print(f"Steering Directions: {len(BF_PARAMS['steering_azimuths_rad'])}")
print(f"Num sensors: {ARRAY_PARAMS['num_sensors']}")
print(f"Sensor Spacing: {ARRAY_PARAMS['sensor_spacing']} m")
print(f"Array length: {ARRAY_PARAMS['num_sensors']*ARRAY_PARAMS['sensor_spacing']} m")
print()
print("Platform Initial State:")
print(f"{tab}Position: ({SHIP_PARAMS['start_vector'][0]:.0f}, "
      f"{SHIP_PARAMS['start_vector'][2]:.0f}) m")
print(f"{tab}Velocity: ({SHIP_PARAMS['start_vector'][1]:.1f}, "
      f"{SHIP_PARAMS['start_vector'][3]:.1f}) m/s")
print()
for target_idx, TARGET_PARAMS in enumerate(TARGETS):
    print(f"Target {target_idx + 1}:")
    print(f"{tab}Frequencies: {TARGET_PARAMS['frequencies_hz']} Hz")
    print(f"{tab}Amplitudes: {20 * np.log10(TARGET_PARAMS['amplitudes_upa'])} "
          f"dB re 1 µPa")
    print(f"{tab}Start Position: ({TARGET_PARAMS['start_vector'][0]:.0f}, "
          f"{TARGET_PARAMS['start_vector'][2]:.0f}) m")
    print(f"{tab}Velocity: ({TARGET_PARAMS['start_vector'][1]:.1f}, "
          f"{TARGET_PARAMS['start_vector'][3]:.1f}) m/s")
    print(f"{tab}Tonal Bandwidth: {TARGET_PARAMS['tonal_bandwidth_hz']:.2f} Hz")
    print(f"{tab}Noise Level: {20 * np.log10(TARGET_PARAMS['noise_amplitude_upa']):.1f}"
          f" dB re 1 µPa")



Random Scenario Summary:
Seed: 2000
Number of Targets: 1
Simulation Duration: 900.0 s across 180 timesteps
Timestep Interval: 5.0 s
Array: 50 sensors, 0.5 m spacing
Signal Sampling Rate: 500.0 Hz
Ambient Noise Level: 48.6 dB re 1 µPa
Steering Directions: 181
Num sensors: 50
Sensor Spacing: 0.5 m
Array length: 25.0 m

Platform Initial State:
    Position: (-2000, 2000) m
    Velocity: (5.0, 0.0) m/s

Target 1:
    Frequencies: [ 90.77693112 118.10627705  36.91860048 127.29258553] Hz
    Amplitudes: [95.55775928 95.46793133 94.32662749 92.0471663 ] dB re 1 µPa
    Start Position: (0, 0) m
    Velocity: (0.0, 8.0) m/s
    Tonal Bandwidth: 0.99 Hz
    Noise Level: 78.9 dB re 1 µPa
